# Two-Stage Hybrid Rocket — Planar 3-DOF Trajectory Simulation

**Simulation phases:** `RAIL (1-DOF) → S1_BURN (3-DOF) → COAST → S2_BURN → S2_COAST`

| Parameter | Stage 1 | Stage 2 |
|-----------|---------|----------|
| Total mass | 73 kg | 58 kg |
| Thrust | 1200 N | 400 N |
| Burn time | 8 s | TBD (config) |

- Launch rail: 7 m, 40° elevation
- Decoupling after S1 burnout

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmartos-br/hybrid-rocket-trajectory/blob/main/two_stage_simulation.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Tuple

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3
})

## 1. Configuration Parameters

Edit these values to match your rocket design.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  LAUNCH SITE & RAIL
# ──────────────────────────────────────────────────────────────────────
g0          = 9.81          # gravitational acceleration [m/s²]
rho         = 1.225         # air density at sea level [kg/m³]
L_rail      = 7.0           # launch rail length [m]
theta_rail  = np.radians(40)  # rail elevation from horizontal [rad] (40°)

# ──────────────────────────────────────────────────────────────────────
#  STAGE 1
# ──────────────────────────────────────────────────────────────────────
m_total_pad = 73.0          # total pad mass (S1 + S2) [kg]
m_s2_total  = 58.0          # S2 total mass at separation [kg]
m_prop1     = 5.0           # S1 propellant mass [kg] (adjust to your motor)
T1          = 1200.0        # S1 thrust (constant) [N]
t_burn1     = 8.0           # S1 burn time [s]

# Derived S1 masses
m_s1_total  = m_total_pad - m_s2_total   # S1 structure + propellant = 15 kg
m_s1_dry    = m_s1_total - m_prop1        # S1 dry (structure only)

# ──────────────────────────────────────────────────────────────────────
#  STAGE 2
# ──────────────────────────────────────────────────────────────────────
m_prop2     = 3.0           # S2 propellant mass [kg]
m_s2_dry    = m_s2_total - m_prop2  # S2 dry mass [kg]
T2          = 400.0         # S2 thrust (constant) [N]
t_burn2     = 6.0           # S2 burn time [s] (adjust to your motor)
sep_delay   = 0.5           # delay between S1 burnout and S2 ignition [s]

# ──────────────────────────────────────────────────────────────────────
#  AERODYNAMICS (shared reference — adjust per stage if needed)
# ──────────────────────────────────────────────────────────────────────
d_ref       = 0.15          # reference diameter [m]
S_ref       = np.pi * (d_ref / 2)**2  # reference area [m²]
CD          = 0.45          # axial drag coefficient (constant)
CNa         = 3.5           # normal force slope [1/rad]
SM          = 1.5           # static margin [calibers] (SM > 0 → stable)
Cmq         = -8.0          # pitch damping coefficient

# ──────────────────────────────────────────────────────────────────────
#  INERTIA (rough cylindrical estimates — replace with CAD values)
# ──────────────────────────────────────────────────────────────────────
L_rocket    = 2.5           # total rocket length [m]
I_s1        = 0.5 * m_s1_dry * (d_ref/2)**2 + m_s1_dry * (L_rocket * 0.3)**2
I_s2        = 0.5 * m_s2_dry * (d_ref/2)**2 + m_s2_dry * (L_rocket * 0.2)**2

# ──────────────────────────────────────────────────────────────────────
#  SIMULATION
# ──────────────────────────────────────────────────────────────────────
dt          = 0.001         # time step [s]
t_max       = 300.0         # max simulation time [s]

print(f"S1 dry mass:   {m_s1_dry:.1f} kg")
print(f"S1 prop mass:  {m_prop1:.1f} kg")
print(f"S2 dry mass:   {m_s2_dry:.1f} kg")
print(f"S2 prop mass:  {m_prop2:.1f} kg")
print(f"Pad mass:      {m_total_pad:.1f} kg")
print(f"S_ref:         {S_ref*1e4:.2f} cm²")
print(f"Rail angle:    {np.degrees(theta_rail):.0f}°")
print(f"T/W ratio S1:  {T1 / (m_total_pad * g0):.2f}")
print(f"T/W ratio S2:  {T2 / (m_s2_total * g0):.2f}")

## 2. Mass & Inertia Model (Variable Mass)

In [ ]:
def propellant_fraction(t_elapsed, t_burn):
    """Linear propellant depletion: f(t) = max(0, 1 - t/t_burn)"""
    return max(0.0, 1.0 - t_elapsed / t_burn)


def get_mass_and_inertia(t, phase):
    """Return (mass, inertia) based on current phase and time."""
    if phase in ('RAIL', 'S1_BURN'):
        f = propellant_fraction(t, t_burn1)
        m = m_s1_dry + m_s2_total + m_prop1 * f
        I = I_s1 + I_s2 + I_s1 * (m_prop1 * f / m_s1_dry)  # scale prop inertia
    elif phase == 'COAST':
        m = m_s1_dry + m_s2_total
        I = I_s1 + I_s2
    elif phase == 'S2_BURN':
        t_s2 = t - (t_burn1 + sep_delay)
        f = propellant_fraction(t_s2, t_burn2)
        m = m_s2_dry + m_prop2 * f
        I = I_s2 + I_s2 * (m_prop2 * f / m_s2_dry)
    else:  # S2_COAST
        m = m_s2_dry
        I = I_s2
    return m, I


def get_thrust(t, phase):
    """Return current thrust based on phase."""
    if phase in ('RAIL', 'S1_BURN'):
        return T1 if t < t_burn1 else 0.0
    elif phase == 'S2_BURN':
        t_s2 = t - (t_burn1 + sep_delay)
        return T2 if t_s2 < t_burn2 else 0.0
    return 0.0

## 3. Phase 0 — Rail (1-DOF, Euler Integration)

Rail is mechanically constrained — motion only along the rail axis.  
`s̈ = (T − D − m·g·sin θ_rail) / m`

In [ ]:
def simulate_rail():
    """Simulate rail phase (1-DOF Euler). Returns rail-exit state."""
    s = 0.0     # position along rail
    V = 0.0     # velocity along rail
    t = 0.0
    max_accel_g = 0.0

    history = {'t': [], 'x': [], 'z': [], 'V': [], 'a_g': []}

    while s < L_rail and t < t_max:
        m, _ = get_mass_and_inertia(t, 'RAIL')
        T = get_thrust(t, 'RAIL')
        D = CD * 0.5 * rho * V**2 * S_ref
        a = (T - D - m * g0 * np.sin(theta_rail)) / m

        accel_g = abs(a) / g0
        max_accel_g = max(max_accel_g, accel_g)

        # Record
        history['t'].append(t)
        history['x'].append(s * np.cos(theta_rail))
        history['z'].append(s * np.sin(theta_rail))
        history['V'].append(V)
        history['a_g'].append(accel_g)

        # Euler step
        V += a * dt
        V = max(V, 0.0)  # no backward motion on rail
        s += V * dt
        t += dt

    # Rail exit conditions
    V_exit = V
    q_exit = 0.5 * rho * V_exit**2
    exit_state = {
        't': t,
        'x': s * np.cos(theta_rail),
        'z': s * np.sin(theta_rail),
        'vx': V_exit * np.cos(theta_rail),
        'vz': V_exit * np.sin(theta_rail),
        'theta': theta_rail,
        'omega': 0.0,
        'V': V_exit,
        'q': q_exit,
        'max_accel_g': max_accel_g,
    }

    print(f"=== RAIL EXIT ===")
    print(f"  t        = {t:.3f} s")
    print(f"  V_exit   = {V_exit:.2f} m/s  ({V_exit*3.6:.1f} km/h)")
    print(f"  q∞       = {q_exit:.1f} Pa")
    print(f"  alt      = {exit_state['z']:.2f} m")
    print(f"  max g    = {max_accel_g:.2f} g")

    return exit_state, history

## 4. Phases 1–4 — Free Flight (3-DOF, RK4)

State vector: `y = [x, z, vx, vz, θ, ω]`

**EoM:**
- `m·ẍ  = T·cosθ − D·(vx/V) − N·sinθ`
- `m·z̈  = T·sinθ − D·(vz/V) + N·cosθ − m·g`
- `I·θ̈  = M_restore + M_damp`

In [ ]:
def get_phase(t):
    """Determine flight phase from elapsed time."""
    if t < t_burn1:
        return 'S1_BURN'
    elif t < t_burn1 + sep_delay:
        return 'COAST'
    elif t < t_burn1 + sep_delay + t_burn2:
        return 'S2_BURN'
    else:
        return 'S2_COAST'


def derivatives(t, y):
    """Compute dy/dt for the 3-DOF state vector."""
    x, z, vx, vz, theta, omega = y

    phase = get_phase(t)
    m, I = get_mass_and_inertia(t, phase)
    T = get_thrust(t, phase)

    V = np.sqrt(vx**2 + vz**2)
    if V < 1e-6:
        V = 1e-6

    gamma = np.arctan2(vz, vx)        # flight path angle
    alpha = theta - gamma              # angle of attack
    q_inf = 0.5 * rho * V**2          # dynamic pressure

    # Forces
    D = CD * q_inf * S_ref             # drag (axial)
    N = CNa * q_inf * S_ref * alpha    # normal force

    # Translational EoM
    ax = (T * np.cos(theta) - D * (vx / V) - N * np.sin(theta)) / m
    az = (T * np.sin(theta) - D * (vz / V) + N * np.cos(theta) - m * g0) / m

    # Pitch moment
    xcp_xcm = SM * d_ref              # static margin in meters
    M_restore = -CNa * q_inf * S_ref * xcp_xcm * alpha
    M_damp = -Cmq * q_inf * S_ref * d_ref**2 / (2 * V) * omega
    alpha_dot = (M_restore + M_damp) / I

    return np.array([vx, vz, ax, az, omega, alpha_dot])


def rk4_step(t, y, dt):
    """Single RK4 integration step."""
    k1 = derivatives(t, y)
    k2 = derivatives(t + dt/2, y + dt/2 * k1)
    k3 = derivatives(t + dt/2, y + dt/2 * k2)
    k4 = derivatives(t + dt, y + dt * k3)
    return y + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)

## 5. Run Full Simulation

In [ ]:
# ─── Phase 0: Rail ─────────────────────────────────────────────────────
rail_exit, rail_hist = simulate_rail()

# ─── Phases 1–4: Free flight ──────────────────────────────────────────────
y = np.array([
    rail_exit['x'],
    rail_exit['z'],
    rail_exit['vx'],
    rail_exit['vz'],
    rail_exit['theta'],
    rail_exit['omega'],
])

t = rail_exit['t']
dt_sim = 0.005  # 5 ms for free-flight (RK4 is more accurate)

# Storage
history = {
    't': list(rail_hist['t']),
    'x': list(rail_hist['x']),
    'z': list(rail_hist['z']),
    'vx': [], 'vz': [],
    'V': list(rail_hist['V']),
    'theta': [], 'alpha': [],
    'phase': [],
    'accel_g': list(rail_hist['a_g']),
    'mach': [],
}

# Pad rail entries for vx/vz/theta/alpha/phase/mach
for i in range(len(rail_hist['t'])):
    v = rail_hist['V'][i]
    history['vx'].append(v * np.cos(theta_rail))
    history['vz'].append(v * np.sin(theta_rail))
    history['theta'].append(np.degrees(theta_rail))
    history['alpha'].append(0.0)
    history['phase'].append('RAIL')
    history['mach'].append(v / 343.0)  # speed of sound ~343 m/s

# Event tracking
events = {'rail_exit': rail_exit}
apogee_found = False
prev_vz = y[3]
s1_burnout_logged = False
s2_ignition_logged = False

print(f"\nStarting free-flight integration at t={t:.3f} s ...")

while t < t_max:
    phase = get_phase(t)

    # Integrate
    y = rk4_step(t, y, dt_sim)
    t += dt_sim

    x, z, vx, vz, theta, omega = y
    V = np.sqrt(vx**2 + vz**2)
    gamma = np.arctan2(vz, vx)
    alpha = theta - gamma
    m, _ = get_mass_and_inertia(t, phase)

    # Store (downsample: every 10 steps)
    if int(t / dt_sim) % 10 == 0:
        history['t'].append(t)
        history['x'].append(x)
        history['z'].append(z)
        history['vx'].append(vx)
        history['vz'].append(vz)
        history['V'].append(V)
        history['theta'].append(np.degrees(theta))
        history['alpha'].append(np.degrees(alpha))
        history['phase'].append(phase)
        history['mach'].append(V / 343.0)

        # Acceleration magnitude
        dy = derivatives(t, y)
        a_mag = np.sqrt(dy[2]**2 + dy[3]**2) / g0
        history['accel_g'].append(a_mag)

    # ── Event: S1 burnout / separation ──
    if not s1_burnout_logged and t >= t_burn1:
        s1_burnout_logged = True
        events['s1_burnout'] = {
            't': t, 'V': V, 'alt': z, 'Mach': V/343,
            'alpha_deg': np.degrees(alpha), 'gamma_deg': np.degrees(gamma)
        }
        print(f"\n=== S1 BURNOUT / SEPARATION ===")
        print(f"  t     = {t:.2f} s")
        print(f"  V     = {V:.1f} m/s  (Mach {V/343:.2f})")
        print(f"  alt   = {z:.1f} m")
        print(f"  γ     = {np.degrees(gamma):.1f}°")

    # ── Event: S2 ignition ──
    if not s2_ignition_logged and t >= t_burn1 + sep_delay:
        s2_ignition_logged = True
        events['s2_ignition'] = {'t': t, 'V': V, 'alt': z, 'Mach': V/343}
        print(f"\n=== S2 IGNITION ===")
        print(f"  t     = {t:.2f} s")
        print(f"  V     = {V:.1f} m/s  (Mach {V/343:.2f})")
        print(f"  alt   = {z:.1f} m")

    # ── Event: Apogee ──
    if not apogee_found and vz < 0 and prev_vz >= 0:
        apogee_found = True
        events['apogee'] = {
            't': t, 'V': V, 'alt': z,
            'downrange': x, 'Mach': V/343
        }
        print(f"\n=== APOGEE ===")
        print(f"  t         = {t:.2f} s")
        print(f"  altitude  = {z:.1f} m  ({z/1000:.2f} km)")
        print(f"  downrange = {x:.1f} m")
        print(f"  V         = {V:.1f} m/s")

    prev_vz = vz

    # ── Termination: ground impact ──
    if z < 0 and t > 1.0:
        events['impact'] = {'t': t, 'x': x, 'V': V}
        print(f"\n=== GROUND IMPACT ===")
        print(f"  t         = {t:.2f} s")
        print(f"  downrange = {x:.1f} m  ({x/1000:.2f} km)")
        print(f"  V_impact  = {V:.1f} m/s")
        break

print(f"\nSimulation complete. {len(history['t'])} data points recorded.")

## 6. Results — Trajectory Plot

In [ ]:
t_arr = np.array(history['t'])
x_arr = np.array(history['x'])
z_arr = np.array(history['z'])
V_arr = np.array(history['V'])
phase_arr = np.array(history['phase'])

phase_colors = {
    'RAIL': '#888888',
    'S1_BURN': '#e74c3c',
    'COAST': '#f39c12',
    'S2_BURN': '#2ecc71',
    'S2_COAST': '#3498db'
}

fig, ax = plt.subplots(figsize=(14, 8))

for phase_name, color in phase_colors.items():
    mask = phase_arr == phase_name
    if mask.any():
        ax.scatter(x_arr[mask] / 1000, z_arr[mask] / 1000,
                   c=color, s=1, label=phase_name, zorder=2)

# Mark events
if 'apogee' in events:
    ap = events['apogee']
    ax.plot(ap['downrange']/1000, ap['alt']/1000, 'r*', ms=15,
            label=f"Apogee: {ap['alt']:.0f} m", zorder=5)

ax.set_xlabel('Downrange [km]')
ax.set_ylabel('Altitude [km]')
ax.set_title('Two-Stage Rocket Trajectory (Planar 3-DOF)')
ax.legend(loc='upper right', fontsize=9)
ax.set_aspect('equal')
ax.set_ylim(bottom=-0.1)
plt.tight_layout()
plt.show()

## 7. Results — Time History Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Phase background shading helper
def shade_phases(ax):
    phase_regions = [
        (0, rail_exit['t'], '#888888', 'Rail'),
        (rail_exit['t'], t_burn1, '#e74c3c', 'S1 Burn'),
        (t_burn1, t_burn1 + sep_delay, '#f39c12', 'Coast'),
        (t_burn1 + sep_delay, t_burn1 + sep_delay + t_burn2, '#2ecc71', 'S2 Burn'),
    ]
    for t0, t1, color, label in phase_regions:
        ax.axvspan(t0, t1, alpha=0.08, color=color)

# ── Altitude vs Time ──
ax1 = axes[0, 0]
ax1.plot(t_arr, z_arr, 'b-', lw=1.2)
shade_phases(ax1)
if 'apogee' in events:
    ax1.axhline(events['apogee']['alt'], ls='--', color='red', alpha=0.5, lw=0.8)
    ax1.annotate(f"Apogee: {events['apogee']['alt']:.0f} m",
                 xy=(events['apogee']['t'], events['apogee']['alt']),
                 fontsize=9, color='red')
ax1.set_xlabel('Time [s]')
ax1.set_ylabel('Altitude [m]')
ax1.set_title('Altitude vs Time')

# ── Velocity vs Time ──
ax2 = axes[0, 1]
ax2.plot(t_arr, V_arr, 'r-', lw=1.2)
shade_phases(ax2)
ax2.set_xlabel('Time [s]')
ax2.set_ylabel('Velocity [m/s]')
ax2.set_title('Velocity Magnitude vs Time')

# ── Mach Number ──
ax3 = axes[1, 0]
mach_arr = np.array(history['mach'])
ax3.plot(t_arr, mach_arr, 'g-', lw=1.2)
ax3.axhline(1.0, ls='--', color='black', alpha=0.4, lw=0.8, label='Mach 1')
shade_phases(ax3)
ax3.set_xlabel('Time [s]')
ax3.set_ylabel('Mach Number')
ax3.set_title('Mach Number vs Time')
ax3.legend(fontsize=9)

# ── Angle of Attack ──
ax4 = axes[1, 1]
alpha_arr = np.array(history['alpha'])
# Clip for readability
alpha_clip = np.clip(alpha_arr, -20, 20)
ax4.plot(t_arr, alpha_clip, 'm-', lw=0.8)
shade_phases(ax4)
ax4.set_xlabel('Time [s]')
ax4.set_ylabel('α [deg]')
ax4.set_title('Angle of Attack vs Time')

plt.tight_layout()
plt.show()

## 8. Results — Acceleration Profile

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

accel_arr = np.array(history['accel_g'])
ax.plot(t_arr, accel_arr, 'k-', lw=1.0)
shade_phases(ax)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Acceleration [g]')
ax.set_title('Acceleration Profile')
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print(f"\nPeak acceleration: {accel_arr.max():.2f} g")

## 9. Summary Table

In [ ]:
print("=" * 60)
print("  TWO-STAGE ROCKET SIMULATION — SUMMARY")
print("=" * 60)

re = events.get('rail_exit', {})
s1 = events.get('s1_burnout', {})
s2i = events.get('s2_ignition', {})
ap = events.get('apogee', {})
imp = events.get('impact', {})

rows = [
    ('Rail exit',      f"{re.get('t',0):.2f} s",   f"{re.get('V',0):.1f} m/s",  f"{re.get('z',0):.1f} m"),
    ('S1 burnout',     f"{s1.get('t',0):.2f} s",   f"{s1.get('V',0):.1f} m/s",  f"{s1.get('alt',0):.1f} m"),
    ('S2 ignition',    f"{s2i.get('t',0):.2f} s",  f"{s2i.get('V',0):.1f} m/s", f"{s2i.get('alt',0):.1f} m"),
    ('Apogee',         f"{ap.get('t',0):.2f} s",   f"{ap.get('V',0):.1f} m/s",  f"{ap.get('alt',0):.1f} m"),
    ('Impact',         f"{imp.get('t',0):.2f} s",  f"{imp.get('V',0):.1f} m/s", f"{imp.get('x',0):.1f} m range"),
]

print(f"{'Event':<16} {'Time':<14} {'Velocity':<16} {'Altitude/Range'}")
print("-" * 60)
for name, t_str, v_str, alt_str in rows:
    print(f"{name:<16} {t_str:<14} {v_str:<16} {alt_str}")

print("\n" + "=" * 60)
if ap:
    print(f"  ★ APOGEE: {ap['alt']:.0f} m ({ap['alt']/1000:.2f} km)")
    print(f"  ★ DOWNRANGE AT APOGEE: {ap.get('downrange',0):.0f} m")
if imp:
    print(f"  ★ TOTAL RANGE: {imp['x']:.0f} m ({imp['x']/1000:.2f} km)")
    print(f"  ★ FLIGHT TIME: {imp['t']:.1f} s")
print("=" * 60)

---

### Simplifications & Next Steps

**Current model assumptions:**
1. Constant CD / CNα — no Mach-dependent aero tables
2. Uniform atmosphere (ρ constant, no altitude variation)
3. Linear propellant depletion (real motors have a thrust curve)
4. Planar only — no wind, yaw, roll, or sideslip
5. Single reference area — both stages share diameter/CD

**Recommended improvements:**
- Add Mach-dependent CD(M) table from OpenRocket / RASAero
- Add ISA atmospheric model: ρ(z), T(z), a(z)
- Import real thrust curve from `.eng` / static fire data
- Separate aero properties per stage (different diameters)
- Monte Carlo over SM, sep_delay, rail angle for dispersion